In [ ]:
from flask import Flask,Request,jsonify
from pymongo import MongoClient
from bson.objectid import ObjectId
import pyodbc
import pandas as pd 
import numpy as np
import threading
import time  
from nltk.tokenize import word_tokenize, sent_tokenize  # for splitting text into words and sentences
from nltk.corpus import stopwords                      # for filtering out common words
from nltk.stem import WordNetLemmatizer                # for reducing words to their base form
import string    
from pymongo.errors import BulkWriteError
import requests


In [ ]:
app=Flask(__name__)

In [ ]:
Client=MongoClient("mongodb://localhost:27017/")
db=Client["FINALPROJECT"]
coll=db["TRYDATA"]
docs = list(coll.find({}))

In [ ]:
from nltk.corpus import wordnet
import nltk
# nltk.download("wordnet")
# nltk.download("omw-1.4")

# ===============================
# Step 1: Build a comprehensive list of "symptom-like" words
# ===============================
def get_synonyms(word):
    """Return a set of synonyms for a word using WordNet"""
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            candidate = lemma.name().replace("_", " ").lower()
            if candidate.isalpha() and len(candidate) > 2:
                synonyms.add(candidate)
    return synonyms

# Base keyword
BASE_SYMPTOM_WORDS = ["symptom", "sign", "manifestation", "indication", "feature", "complaint"]

SYMPTOM_KEYWORDS = set()
for w in BASE_SYMPTOM_WORDS:
    SYMPTOM_KEYWORDS.add(w)
    SYMPTOM_KEYWORDS.update(get_synonyms(w))

# Add manually other common clinical expressions
MANUAL_KEYWORDS = [
    "presentation", "clinical finding", "phenomenon", "issue", "abnormality",
    "condition", "experience", "effect", "expression", "signal", "cue"
]
SYMPTOM_KEYWORDS.update(MANUAL_KEYWORDS)

print(f"Total symptom keywords: {len(SYMPTOM_KEYWORDS)}")

# ===============================
# Step 2: Recursive text extractor
# ===============================
def extract_text(data):
    texts = []
    if isinstance(data, str):
        t = data.strip()
        if t:
            texts.append(t)
    elif isinstance(data, dict):
        for v in data.values():
            texts.extend(extract_text(v))
    elif isinstance(data, list):
        for item in data:
            texts.extend(extract_text(item))
    return texts

# ===============================
# Step 3: Extract symptoms smartly from docs
# ===============================
def extract_symptoms(docs):
    result_symptoms = []

    for doc in docs:
        name = doc.get("name", "")
        symptoms = []

        for section in doc.get("sections", []):
            title = section.get("title", "").strip().lower()
            paragraphs = section.get("paragraphs", [])

            # Check if title contains any symptom keyword
            title_flag = any(keyword in title for keyword in SYMPTOM_KEYWORDS)

            # Also scan paragraph content for any symptom keyword
            paragraph_text = " ".join(extract_text(paragraphs)).lower()
            paragraph_flag = any(keyword in paragraph_text for keyword in SYMPTOM_KEYWORDS)

            if title_flag or paragraph_flag:
                symptoms.extend(extract_text(paragraphs))

        if symptoms:
            result_symptoms.append({
                "name": name,
                "Symptoms": symptoms
            })

    print("Total extracted records:", len(result_symptoms))
    return result_symptoms 

In [ ]:
result_symptoms = extract_symptoms(docs)

# عرض النتائج
for r in result_symptoms:
    print("Disease:", r["name"])
    print("Symptoms:", r["Symptoms"])
    print("---")


In [ ]:
# =========================================================
# Clinical Hybrid Medical AI Agent
# Hybrid Retrieval + LLM Guidance (BioBERT)
# =========================================================

import os, json, random, re, hashlib
from collections import defaultdict

# NLP
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import wordnet
from nltk import pos_tag

# Embeddings
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# LLM
import google.generativeai as genai

# UI
import gradio as gr

# =========================================================
# SETTINGS
# =========================================================
TARGET_RECORDS = 20
MAX_TRIES_PER_DISEASE = 200
BATCH_SIZE = 10
REPLACE_RATIO = 0.2
MEDICAL_BOOST = 1.3
AMBIGUITY_MARGIN_RATIO = 0.1
LOW_CONF_THRESHOLD = 0.45
TOP_K_RETRIEVE = 10

MEDICAL_KEEP = {
    "pain","fever","rash","swelling","headache","nausea","vomiting",
    "cough","fatigue","abdomen","abdominal","epigastric",
    "right","left","lower","upper","back","radiating",
    "acute","chronic","constant","severe","mild",
    "alcohol","meal","fatty","diarrhea","breath","shortness"
}

BAD_WORDS = {
    "siemens","angstrom","beryllium","nobelium",
    "atomic number","terminated time","oregon"
}

# =========================================================
# LLM CONFIG
# =========================================================
import html
from dotenv import load_dotenv
load_dotenv()  # reads GEMINI_API_KEY from a local .env file (never commit .env)
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("GEMINI_API_KEY is not set. Copy .env.example to .env and add your key.")
genai.configure(api_key=API_KEY)
llm_model = genai.GenerativeModel(
    model_name="models/gemini-2.5-flash",
    generation_config={"temperature":0.2, "max_output_tokens":300}
)

# =========================================================
# EMBEDDING MODEL (BioBERT)
# =========================================================
embed_model = SentenceTransformer("pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb")

# =========================================================
# HELPERS
# =========================================================
def normalize_text(text):
    if isinstance(text, list):
        text = " ".join(text)
    return str(text).strip()

def fast_hash(text):
    return hashlib.md5(text.lower().encode()).hexdigest()

def is_good_text(text):
    t = text.lower()
    if any(w in t for w in BAD_WORDS):
        return False
    if len(text.split()) < 5:
        return False
    letters = re.findall(r"[a-zA-Z]", text)
    if len(letters)/max(1,len(text)) < 0.75:
        return False
    return True

# =========================================================
# SAFE SYNONYM AUGMENTATION
# =========================================================
def safe_synonym(word, pos):
    if word.lower() in MEDICAL_KEEP:
        return word
    if pos.startswith("N"): wn_pos = wordnet.NOUN
    elif pos.startswith("V"): wn_pos = wordnet.VERB
    elif pos.startswith("J"): wn_pos = wordnet.ADJ
    else: return word
    syns = wordnet.synsets(word, pos=wn_pos)
    for s in syns:
        for lemma in s.lemmas():
            candidate = lemma.name().replace("_"," ")
            if candidate.isalpha() and len(candidate) > 3:
                return candidate
    return word

def augment_sentence(sentence):
    words = word_tokenize(sentence)
    tags = pos_tag(words)
    idxs = [i for i,w in enumerate(words) if w.isalpha() and w.lower() not in MEDICAL_KEEP]
    if not idxs: return sentence
    num_replace = max(1,int(len(idxs)*REPLACE_RATIO))
    chosen = random.sample(idxs,min(num_replace,len(idxs)))
    new_words = words[:]
    for i in chosen: new_words[i] = safe_synonym(words[i], tags[i][1])
    return " ".join(new_words)

def augment_text(text):
    sentences = sent_tokenize(text)
    return " ".join(augment_sentence(s) for s in sentences if len(s) > 3)

# =========================================================
# LOAD DATA + AUGMENTATION
# =========================================================
final_data = []
name_counts = defaultdict(int)

# result_symptoms يجب أن يكون متوفر كـ list of dict {"name": "disease", "Symptoms": "text"}
for item in result_symptoms:
    name = item["name"]
    base_text = normalize_text(item["Symptoms"])
    if not is_good_text(base_text):
        continue
    final_data.append({"name": name, "symptoms": base_text})
    name_counts[name] += 1
    hashes = {fast_hash(base_text)}
    tries = 0
    while name_counts[name] < TARGET_RECORDS and tries < MAX_TRIES_PER_DISEASE:
        tries += 1
        source_text = random.choice([x["symptoms"] for x in final_data if x["name"]==name])
        for _ in range(BATCH_SIZE):
            if name_counts[name]>=TARGET_RECORDS: break
            new_text = augment_text(source_text)
            if not is_good_text(new_text): continue
            h = fast_hash(new_text)
            if h in hashes: continue
            final_data.append({"name":name,"symptoms":new_text})
            hashes.add(h)
            name_counts[name]+=1

print("Total records:", len(final_data))

# =========================================================
# EMBEDDINGS + FAISS INDEX
# =========================================================
texts = [x["symptoms"] for x in final_data]
labels = [x["name"] for x in final_data]

if not texts:
    raise ValueError("No valid symptom data found. Check your result_symptoms.")

embeddings = embed_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
embeddings = np.array(embeddings)
if embeddings.ndim == 1:
    embeddings = embeddings.reshape(1, -1)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

# =========================================================
# IMPROVED SIMILARITY SEARCH + RERANKING
# =========================================================
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-10))

def get_weighted_embedding(text):
    tokens = word_tokenize(text.lower())
    token_embeddings = embed_model.encode(tokens, convert_to_numpy=True, normalize_embeddings=True)
    token_embeddings = np.array(token_embeddings)
    if token_embeddings.ndim == 1:
        token_embeddings = token_embeddings.reshape(1, -1)
    weights = np.array([MEDICAL_BOOST if w in MEDICAL_KEEP else 1.0 for w in tokens]).reshape(-1,1)
    weighted_emb = (token_embeddings * weights).sum(axis=0) / weights.sum()
    weighted_emb /= np.linalg.norm(weighted_emb) + 1e-10
    return weighted_emb

def similarity_search_rerank(query, top_k=5):
    query_emb = get_weighted_embedding(query).reshape(1, -1)

    # Step 1: retrieve top-K from FAISS
    D,I = index.search(query_emb, TOP_K_RETRIEVE)
    candidates = []
    for score, idx in zip(D[0], I[0]):
        candidates.append({"name": labels[idx], "faiss_score": float(score), "symptoms": texts[idx]})

    # Step 2: rerank semantic based on full weighted embedding
    for c in candidates:
        c_emb = get_weighted_embedding(c["symptoms"])
        c["score"] = cosine_sim(query_emb.flatten(), c_emb)

    # Step 3: aggregate by disease name, take max similarity
    disease_scores = defaultdict(float)
    for c in candidates:
        disease_scores[c["name"]] = max(disease_scores[c["name"]], c["score"])

    results = [{"name":d,"score":round(s,4)} for d,s in disease_scores.items()]
    results.sort(key=lambda x:x["score"], reverse=True)
    results = results[:top_k]

    # Step 4: dynamic ambiguity margin
    status = "confident"
    if len(results) > 1:
        diff = results[0]["score"] - results[1]["score"]
        if diff < results[0]["score"] * AMBIGUITY_MARGIN_RATIO:
            status = "ambiguous"

    if results and results[0]["score"] < LOW_CONF_THRESHOLD:
        status = "low_confidence"

    return {"type": status, "candidates": results}

# =========================================================
# LLM GUIDANCE AGENT
# =========================================================
class LLM_Guidance_Agent:
    def __init__(self, model):
        self.model = model

    def get_guidance(self, disease_name):
        prompt=f"""
You are a medical guidance assistant.
Patient has: {disease_name}
Provide 4-6 immediate non-diagnostic actions/advice.
Return JSON only:
{{"actions":["Action1","Action2"]}}
"""
        try:
            r = self.model.generate_content(prompt)
            t = r.text.strip()
            j = t[t.find("{"):t.rfind("}")+1]
            return json.loads(j)["actions"]
        except:
            return [
                "Seek medical attention",
                "Rest and avoid exertion",
                "Do not self-medicate",
                "Monitor symptoms closely"
            ]

# =========================================================
# MAIN AGENT LOGIC
# =========================================================
def interactive_agent(symptoms, clarifying_answers=[]):
    agent = LLM_Guidance_Agent(llm_model)
    query = symptoms
    if clarifying_answers:
        query += " " + " ".join(clarifying_answers)
    retrieval = similarity_search_rerank(query, top_k=3)
    candidates = retrieval["candidates"]

    if retrieval["type"] == "low_confidence":
        return "<h2 style='color:red'>❗ Low Confidence. Consult a doctor.</h2>", []

    if retrieval["type"] == "confident":
        disease = candidates[0]["name"]
        guidance = agent.get_guidance(disease)
        return f"""
<h2 style='color:green'>✅ Confident Diagnosis</h2>
<h3>Disease: {html.escape(str(disease))}</h3>
<ul>{''.join([f'<li>{html.escape(str(a))}</li>' for a in guidance])}</ul>
<h4>Similarity Scores:</h4>
<ul>{''.join([f'<li>{c["score"]} => {html.escape(str(c["name"]))}</li>' for c in candidates])}</ul>
""", []

    # ambiguous
    clarifying_questions = [f"Do you have symptoms specific to {c['name']}?" for c in candidates]
    return f"""
<h2 style='color:orange'>⚠️ Ambiguous Diagnosis</h2>
<ul>{''.join([f'<li>{html.escape(str(c["name"]))} ({c["score"]})</li>' for c in candidates])}</ul>
""", clarifying_questions

# =========================================================
# GRADIO UI
# =========================================================
with gr.Blocks() as demo:
    gr.Markdown("## 🩺 Hybrid Clinical AI Agent (BioBERT)")
    symptoms_input = gr.Textbox(label="Enter symptoms", lines=3)
    clarifying_answers = gr.State([])
    output_html = gr.HTML()
    clarifying_box = gr.Textbox(label="Clarifying questions", lines=3)

    def submit(symptoms, answers):
        html, questions = interactive_agent(symptoms, answers)
        return html, "\n".join(questions)

    btn = gr.Button("Submit")
    btn.click(submit, inputs=[symptoms_input, clarifying_answers],
              outputs=[output_html, clarifying_box])

demo.launch(share=os.getenv("GRADIO_SHARE", "false").lower() == "true")
